# **Objective: To analyze sales patterns, demand behavior, and price sensitivity across products, and derive actionable pricing and revenue strategies using elasticity-based segmentation.**

***dataset used :- online-retail-ii-uci.zip, 2 years retail data of a certain company that have sales accross several countries but 90% from United Kindom. Variables include Quantity, Price variations, Description of product / name of product and some unneccsary columns that removed further in this notebook***

In [0]:
%pip install kaggle


In [0]:
!kaggle datasets download -d mashlyn/online-retail-ii-uci



In [0]:
!unzip online-retail-ii-uci.zip

In [0]:
# moving the dataset into base folder for better call

import shutil

shutil.move(
"/Workspace/Users/addysiddiqui7@gmail.com/The Logistic Project/online_retail_II.csv",
"/Workspace/online_retail_II.csv"
)

In [0]:
df = spark.read.csv("/Workspace/online_retail_II.csv", header=True, inferSchema=True)
display(df.limit(2))


In [0]:
display(df.describe())

# total of 144867 rows are in our dataset

In [0]:
# removing negative Quanitity columns as the scope of this project is not return analysis.
df = df.filter((df.Quantity > 0) & (df.Price > 0))


In [0]:
display(df.head(10))

In [0]:
dex1 = df.filter(df.Description == "PINK CHERRY LIGHTS")
display(dex1)


# stockcode is identical for every single unit so be dropped further

In [0]:
#dropping unncessarry columns that don't have any use in scope of this project...

df = df.drop("StockCode","Invoice", "Customer ID")

In [0]:
from pyspark.sql.functions import min, max

display(
    df.select(
        min("InvoiceDate").alias("start_date"),
        max("InvoiceDate").alias("end_date")
    )
)

#dataset start date is 01/12/2009 and end date is 09/12/2009. Which means its a 2 year continuous dataset.

In [0]:
display(df.head(5))

In [0]:
df.write.format("delta").saveAsTable("retail")
# conclusion: there are 5399 total unique products purchased accross 43 countries in span of two years.

## **A quick summary of dataset...**

In [0]:
%sql
SELECT
    COUNT(DISTINCT Description) AS unique_products,
    SUM(Quantity) AS total_quantity_sold,
    SUM(Price) AS total_price,

    (SELECT Description
     FROM retail
     GROUP BY Description
     ORDER BY SUM(Quantity) DESC
     LIMIT 1) AS most_purchased_product,

    (SELECT Description
     FROM retail
     GROUP BY Description
     ORDER BY SUM(Quantity) ASC
     LIMIT 1) AS least_purchased_product

FROM retail

### Total of 5399 unique products being purchased.        Total quantity sold 11,420,306 units at a total price of 4,245,932 units. The most purchased product was WORLD WAR 2 GLIDERS and Least purchased was a CRYSTAL SMALL JWELLED PHOTOFRAME.